# Run BERT and RoBERTa on a free GPU

Automated retraining can't start a Colab or Kaggle notebook, so these two retrain here when an alert asks (DEC-022). The code is the same `trainer.train` the laptop runs, so a run from here compares directly with one from anywhere else.

Before running:
1. Choose a GPU runtime.
2. Add `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, and `MLFLOW_TRACKING_PASSWORD` as Colab secrets (Kaggle: Add-ons, Secrets). Never paste a token into a cell, because a notebook gets committed.
3. Make sure the code you want is pushed to GitHub, because the next cells clone `main`.

**`FULL_RUN = False` is a test.** It proves the setup works by running the sanity checks on RoBERTa, and sends nothing to DagsHub. `FULL_RUN = True` trains both models for hours, and uploads about 630 MB to DagsHub for each one. A free session can disconnect partway, and its files go with it.

In [ ]:
import os
import subprocess

FULL_RUN = False

try:
    from google.colab import userdata  # Colab

    get = userdata.get
except ImportError:
    from kaggle_secrets import UserSecretsClient  # Kaggle

    get = UserSecretsClient().get_secret
for name in ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    os.environ[name] = get(name)

The project pins Python 3.11, so this builds its own 3.11 environment and doesn't depend on the runtime's Python version. The last line stops here if there's no GPU.

In [ ]:
!git clone --depth 1 https://github.com/AndrewRaj0510/WikiGuard.git
%cd WikiGuard
!pip install -q uv
!uv venv --python 3.11 --seed .venv
!.venv/bin/python -m pip install -q -r jobs/trainer/requirements.txt
!.venv/bin/python -c "import torch; print(torch.__version__, torch.cuda.get_device_name(0))"

Fetch the snapshot from DagsHub. It needs a login even though the repo is public (F-093), so the token goes in `.dvc/config.local`, which git ignores. Set `SNAPSHOT` to the name of the newest pointer in `snapshots/`.

In [ ]:
SNAPSHOT = "snapshots/snapshot-20261104-28c825af4b19"
for key in ("access_key_id", "secret_access_key"):
    token = os.environ["MLFLOW_TRACKING_PASSWORD"]
    # Output is captured, and the check doesn't print the command, so the token stays off screen.
    done = subprocess.run(  # noqa: S603
        [".venv/bin/dvc", "remote", "modify", "origin", "--local", key, token],
        capture_output=True,
        check=False,
    )
    if done.returncode:
        raise RuntimeError(f"DVC couldn't save {key}")
!.venv/bin/dvc pull {SNAPSHOT}.dvc

In [ ]:
!.venv/bin/python scripts/validate_snapshot.py {SNAPSHOT} --skip-evidently

The test logs only to this session, so it adds nothing to DagsHub. A full run chooses a learning rate on validation, trains on the full training window, exports the int8 model, and logs everything to MLflow. Registering happens later, through the gate, so nothing here can change what's being served.

In [ ]:
if FULL_RUN:
    !.venv/bin/python -m trainer.train {SNAPSHOT} --model bert --search
    !.venv/bin/python -m trainer.train {SNAPSHOT} --model roberta --search
else:
    os.environ.pop("MLFLOW_TRACKING_URI")
    !.venv/bin/python -m trainer.sanity {SNAPSHOT} --models roberta